In [6]:
import os 
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

PLOT_DIR = os.path.join(os.path.abspath(os.getcwd()), 'plots/') 
sns.set_style('darkgrid')
  
# print(file_name.split('/')[-2] + '/' + file_name.split('/')[-1])
def show_histoplot(file_name, model, preprocess_label='', algorithm_label=None):
    path = os.path.join(os.path.abspath(os.getcwd()), file_name) 
    # data = []
    # with open(path, 'r') as f:
    #     accuracy = float(f.readline().strip().split(','))
    #     for line in f:
    #         data.append(float(line.strip()))
            
    # Read the CSV file with a comma as the separator
    df = pd.read_csv(path, sep=',')
    # take first row as correct accuracy and remove it
    best_model = df.iloc[0] 
    df = df.drop(0)
    # print(f'Best model for {model}: {best_model.to_dict()}')
    
    # Iterate over all columns and plot the histogram            
    
    metrics_p = {}
    for column in df.columns:
        # print(column)
        data = df[column]
        # print(data)
        # # Count how many Nan values are in the column
        # nan_count = data.isna().sum()
        # print(f'Column {column} has {nan_count} NaN values')
        # # Remove the NaN values
        # data = data.dropna()
        metric = best_model[column]
        
        # plt.figure(figsize=(12, 8))
        # sns.set_context("paper", font_scale=2)  # Adjust font scale for better readability
        # sns.set_style("whitegrid")       

        # sns.histplot(data, binwidth=0.01, kde=True, linewidth=3)
        # if column == 'Q2':
        #     plt.xlim(-0.25, 0.75)
        # else:
        #     plt.xlim(0, 1)
        # plt.xlabel(f'{column} score')
        
        # # Write the line and the value of the accuracy
        # plt.axvline(metric, color='r', linestyle='dashed', linewidth=3)
        # plt.text(metric, 0.9 * plt.ylim()[1], f' {metric:.2f}', color='r')
        
        # # Save the plot
        # name = file_name.split('/')[-2] + '/' + file_name.split('/')[-1]
        # path_plot = os.path.join(PLOT_DIR, preprocess_type,  name.split('.')[0] + '/')
        # if not os.path.exists(path_plot):
        #     os.makedirs(path_plot)
            
        # # Add labels and title
        # plt.xlabel(f'{column} Score',fontsize=26)
        # plt.ylabel("Occurrences",fontsize=26)
        # plt.rc('xtick',labelsize=24)
        # plt.rc('ytick',labelsize=24)
        # # plt.grid(False)
        # plt.title("p-value", fontsize=26)
        
        # # plt.savefig(path_plot + column + '.png')
        # plt.savefig(path_plot + column + '.pdf')
        
        # plt.title(f'{column} Permutation Test')
        # # plt.savefig(path_plot + column + 'titled_.png')
        # plt.savefig(path_plot + column + 'titled_.pdf')
        # # plt.show()
        # plt.close()
        
        # calculate p-value
        p_sum = sum([1 for x in data if x >= metric])
        p = p_sum / len(data)
        metrics_p[column] = p
        
    algorithm_name = algorithm_label if algorithm_label is not None else model.replace("_", "\\_")
    preprocessing_name = preprocess_label.replace("_", "\\_") if preprocess_label else ""
    return f'{preprocessing_name} & {algorithm_name} & {metrics_p["Accuracy"]:.2f} & {metrics_p["Recall"]:.2f} & {metrics_p["Precision"]:.2f} & {metrics_p["F1"]:.2f} \\\\'
    

In [7]:
import glob
def run_all_permutation_tests(base_dir, preprocess_type, model_names, display_preprocess_label):
    for index, (folder_name, algorithm_label) in enumerate(model_names):
        folder = os.path.join(base_dir, preprocess_type, folder_name)
        files = sorted(glob.glob(os.path.join(folder, 'permutation_test_*.csv')))
        if not files:
            print(f"No permutation test file found for {folder_name} in {folder}")
            continue
        row = show_histoplot(
            files[0],
            folder_name,
            preprocess_label=display_preprocess_label if index == 0 else '',
            algorithm_label=algorithm_label,
        )
        print(row)
    print(r"\hline")

preprocess_labels = {
    '10_SG_MSC': 'SG + MSC',
    '10_SG_SVN': 'SG + SVN',
    '10_SG1_MSC': 'SG1 + MSC',
    '10_SG1_SVN': 'SG1 + SVN',
}

model_names = [
    ('PLS', 'PLS-DA'),
    ('ELM', 'ELM'),
    ('XGBoost', 'XGB'),
    ('RandomForest', 'RF'),
    ('SVM', 'SVM'),
    ('CARS', 'CARS'),
    ('BOSS', 'BOSS'),
    ('GA-iPLS', 'GA-iPLS'),
    ('GA-iPLS_BOSS', 'iGA-BOSS'),
]

print(r"""
\begin{table}[H]
\centering
\renewcommand{\arraystretch}{1.2}
\small
\begin{tabular}{cccccc}
\hline
\textbf{Preprocessing} & \textbf{Algorithm} & \textbf{Accuracy} & \textbf{Recall} & \textbf{Precision} & \textbf{F1} \\
\hline
""")

for method in ['10_SG_MSC', '10_SG_SVN', '10_SG1_MSC', '10_SG1_SVN']:
    run_all_permutation_tests('.', method, model_names, preprocess_labels[method])
    print()

print(r"""
\end{tabular}

\caption{Permutation test p-values associated with each model configuration. Values below 0.01 indicate statistically significant performance improvements compared to chance.}
\label{tab:results_p}
\end{table}
""")


\begin{table}[H]
\centering
\renewcommand{\arraystretch}{1.2}
\small
\begin{tabular}{cccccc}
\hline
\textbf{Preprocessing} & \textbf{Algorithm} & \textbf{Accuracy} & \textbf{Recall} & \textbf{Precision} & \textbf{F1} \\
\hline

SG + MSC & PLS-DA & 0.01 & 0.03 & 0.01 & 0.00 \\
 & ELM & 0.00 & 0.00 & 0.00 & 0.00 \\
 & XGB & 0.01 & 0.03 & 0.02 & 0.01 \\
 & RF & 0.01 & 0.02 & 0.01 & 0.00 \\
 & SVM & 0.01 & 0.03 & 0.02 & 0.01 \\
 & CARS & 0.00 & 0.00 & 0.00 & 0.00 \\
 & BOSS & 0.00 & 0.00 & 0.00 & 0.00 \\
 & GA-iPLS & 0.00 & 0.00 & 0.00 & 0.00 \\
 & iGA-BOSS & 0.00 & 0.00 & 0.00 & 0.00 \\
\hline

SG + SVN & PLS-DA & 0.00 & 0.00 & 0.00 & 0.00 \\
 & ELM & 0.00 & 0.00 & 0.01 & 0.00 \\
 & XGB & 0.00 & 0.01 & 0.00 & 0.00 \\
 & RF & 0.10 & 0.28 & 0.10 & 0.15 \\
 & SVM & 0.00 & 0.01 & 0.00 & 0.00 \\
 & CARS & 0.00 & 0.00 & 0.00 & 0.00 \\
 & BOSS & 0.00 & 0.00 & 0.00 & 0.00 \\
 & GA-iPLS & 0.00 & 0.00 & 0.00 & 0.00 \\
 & iGA-BOSS & 0.00 & 0.00 & 0.00 & 0.00 \\
\hline

SG1 + MSC & PLS-DA & 0.00 & 0